# AME 5003 — Principles of Natural Language Processing
## Lab 3: Information Retrieval with Incidence Matrices and Inverted Indexes

**Manipal School of Information Sciences (MSIS)**  
**Manipal Academy of Higher Education (MAHE)**

### Topics
**Term–document incidence matrix · Boolean retrieval · Inverted index construction · Query processing**

### What we will build

We will start with a small collection of restaurant descriptions and gradually build a simple search system.

The flow is:

**documents → index terms → incidence matrix → Boolean search → inverted index → query processing**

The important goal is to understand **why each representation is needed**, not to write a large amount of code.

## Learning outcomes

After completing this practical, you should be able to:

1. explain and build a **term–document incidence matrix**;
2. use the matrix for simple Boolean **AND, OR and AND NOT** retrieval;
3. explain why a large incidence matrix is sparse;
4. construct an **inverted index**;
5. identify a term's **postings list** and **document frequency (df)**;
6. process simple Boolean queries using postings lists;
7. run a small end-to-end Boolean search system.

> Work through the notebook in order. Each new section uses the idea developed in the previous section.

## 0. Setup

We use:

- `re` for simple token extraction;
- `unicodedata` for consistent Unicode text;
- `defaultdict` for building the inverted index;
- `pandas` only to display the incidence matrix neatly.

Run the setup cell.

In [1]:
import re
import unicodedata
from collections import defaultdict

import pandas as pd

## 0.1 Our document collection

Each restaurant description is treated as one **document**.

Before writing any search code, read the six documents.

### First prediction

Which document(s) should match:

> **affordable AND vegetarian**

In [2]:
documents = {
    "D1": "Coastal Café near Malpe Beach serves affordable vegetarian meals.",
    "D2": "Green Bowl in Manipal serves vegetarian and vegan food.",
    "D3": "Beach Shack near Malpe Beach serves seafood and snacks.",
    "D4": "Campus Bistro near MAHE offers affordable coffee and sandwiches.",
    "D5": "Spice Garden in Udupi serves vegetarian meals and seafood.",
    "D6": "Ocean View Café near the beach offers coffee and seafood."
}

for doc_id, text in documents.items():
    print(f"{doc_id}: {text}")

D1: Coastal Café near Malpe Beach serves affordable vegetarian meals.
D2: Green Bowl in Manipal serves vegetarian and vegan food.
D3: Beach Shack near Malpe Beach serves seafood and snacks.
D4: Campus Bistro near MAHE offers affordable coffee and sandwiches.
D5: Spice Garden in Udupi serves vegetarian meals and seafood.
D6: Ocean View Café near the beach offers coffee and seafood.


# Part 1 — Prepare the text for indexing

The search system should process documents and queries in the **same way**.

For this lab we use a simple policy:

1. Unicode normalization using NFC;
2. lowercase;
3. extract alphabetic word-like tokens;
4. remove only the word `the`.

The preprocessing function is already provided because the focus of this lab is **information retrieval**, not text preprocessing.

In [3]:
def preprocess(text):
    text = unicodedata.normalize("NFC", text)
    text = text.lower()

    tokens = re.findall(r"[^\W\d_]+", text, flags=re.UNICODE)
    tokens = [token for token in tokens if token != "the"]

    return tokens


processed_documents = {
    doc_id: preprocess(text)
    for doc_id, text in documents.items()
}

for doc_id, tokens in processed_documents.items():
    print(doc_id, "->", tokens)

D1 -> ['coastal', 'café', 'near', 'malpe', 'beach', 'serves', 'affordable', 'vegetarian', 'meals']
D2 -> ['green', 'bowl', 'in', 'manipal', 'serves', 'vegetarian', 'and', 'vegan', 'food']
D3 -> ['beach', 'shack', 'near', 'malpe', 'beach', 'serves', 'seafood', 'and', 'snacks']
D4 -> ['campus', 'bistro', 'near', 'mahe', 'offers', 'affordable', 'coffee', 'and', 'sandwiches']
D5 -> ['spice', 'garden', 'in', 'udupi', 'serves', 'vegetarian', 'meals', 'and', 'seafood']
D6 -> ['ocean', 'view', 'café', 'near', 'beach', 'offers', 'coffee', 'and', 'seafood']


### **Understand the Regex expression**

Search documentation and understand the expression used in the cell above.

**Note:**
**`flags=re.UNICODE`** enables Unicode-aware matching, allowing words containing characters such as `é` in *café* to be recognized correctly. In Python 3, this behavior is generally the default, but specifying the flag makes the intent more clear.

### Quick check

Run:

```python
preprocess("Affordable, VEGETARIAN!")
```

You should obtain the same basic terms that would be stored in the index.

This is why **document preprocessing and query preprocessing must be consistent**.

In [4]:
print(preprocess("Affordable, VEGETARIAN!"))

['affordable', 'vegetarian']


# Part 2 — Term–document incidence matrix

A **term–document incidence matrix** stores only **presence or absence**.

- rows = terms
- columns = documents
- `1` = term occurs in the document
- `0` = term does not occur

For example, let us build only the row for `vegetarian`.

In [5]:
term = "vegetarian"
row = []

for doc_id in documents:
    if term in processed_documents[doc_id]:
        row.append(1)
    else:
        row.append(0)

print("Documents :", list(documents.keys()))
print(term, ":", row)

Documents : ['D1', 'D2', 'D3', 'D4', 'D5', 'D6']
vegetarian : [1, 1, 0, 0, 1, 0]


The loop checks the same question for every document:

> **Is `vegetarian` present in this document?**

If yes, append `1`; otherwise append `0`.

Notice that we are **not counting how many times** the term occurs.  
The incidence matrix records only presence or absence.

## Activity 2.1 — Build one more row

Use the same logic for the term:

> `coffee`

Only change the value of `term`.

Before running the code, predict the row.

In [6]:
term = "coffee"
row = []

for doc_id in documents:
    if term in processed_documents[doc_id]:
        row.append(1)
    else:
        row.append(0)

print("Documents :", list(documents.keys()))
print(term, ":", row)

Documents : ['D1', 'D2', 'D3', 'D4', 'D5', 'D6']
coffee : [0, 0, 0, 1, 0, 1]


## 2.2 Build the complete incidence matrix

Now repeat the same idea for several vocabulary terms.

We will use:

`affordable, vegetarian, vegan, seafood, beach, coffee`

### How to think about the code

For **each vocabulary term**:

1. create an empty row;
2. visit each document;
3. check whether the term occurs;
4. append `1` or `0`;
5. save the completed row.

You already wrote the inner logic in Activity 2.1.

### Your task

Complete only the marked line inside the `if` block.

In [7]:
vocabulary = [
    "affordable",
    "vegetarian",
    "vegan",
    "seafood",
    "beach",
    "coffee"
]

matrix_data = {}

for term in vocabulary:
    row = []

    for doc_id in documents:
        if term in processed_documents[doc_id]:
            # TODO: append 1
            row.append(1)
        else:
            row.append(0)

    matrix_data[term] = row

incidence_df = pd.DataFrame(
    matrix_data,
    index=list(documents.keys())
).T

incidence_df

,D1,D2,D3,D4,D5,D6
affordable,1,0,0,1,0,0
vegetarian,1,1,0,0,1,0
vegan,0,1,0,0,0,0
seafood,0,0,1,0,1,1
beach,1,0,1,0,0,1
coffee,0,0,0,1,0,1


## 2.3 Read the matrix

A **row** answers:

> Which documents contain this term?

A **column** answers:

> Which selected vocabulary terms occur in this document?

Use the following code to inspect both.

In [8]:
print("ROW: vegetarian")
print(incidence_df.loc["vegetarian"])

print("\nCOLUMN: D3")
print(incidence_df["D3"])

ROW: vegetarian
D1    1
D2    1
D3    0
D4    0
D5    1
D6    0
Name: vegetarian, dtype: int64

COLUMN: D3
affordable    0
vegetarian    0
vegan         0
seafood       1
beach         1
coffee        0
Name: D3, dtype: int64


### Activity 2.2 — Read, do not code

Using the displayed matrix, answer:

1. Which documents contain `vegetarian`?
2. Which documents contain `coffee`?
3. Which selected terms occur in D3?
4. Why is a repeated word still represented only by `1`?

# Part 3 — Boolean retrieval using the incidence matrix

We can now search the matrix.

For the query:

> `vegetarian AND beach`

a document should be returned only when **both rows contain 1** in the same column.

We will first do one complete example.

In [9]:
row1 = incidence_df.loc["vegetarian"]
row2 = incidence_df.loc["beach"]

and_mask = (row1 == 1) & (row2 == 1)

matching_docs = incidence_df.columns[and_mask].tolist()

print("vegetarian AND beach ->", matching_docs)

vegetarian AND beach -> ['D1']


The important line is:

```python
(row1 == 1) & (row2 == 1)
```

`&` means **AND**.

A document is selected only when both conditions are true.

## 3.1 OR and AND NOT

The same idea works for other Boolean operations.

- `&` means AND
- `|` means OR
- AND NOT means: first term is present **and** second term is absent

The code below is almost complete.

In [10]:
# OR example: coffee OR vegan
row1 = incidence_df.loc["coffee"]
row2 = incidence_df.loc["vegan"]

or_mask = (row1 == 1) | (row2 == 1)
print("coffee OR vegan ->",
      incidence_df.columns[or_mask].tolist())


# AND NOT example: vegetarian AND NOT seafood
row1 = incidence_df.loc["vegetarian"]
row2 = incidence_df.loc["seafood"]

and_not_mask = (row1 == 1) & (row2 == 0)
print("vegetarian AND NOT seafood ->",
      incidence_df.columns[and_not_mask].tolist())

coffee OR vegan -> ['D2', 'D4', 'D6']
vegetarian AND NOT seafood -> ['D1', 'D2']


### Activity 3.1 — Change only the query terms

Use the same patterns above to find:

1. `affordable AND vegetarian`
2. `seafood AND beach`
3. `beach AND NOT vegetarian`

Do **not** write a new function.  
Just change the term names and use the appropriate Boolean pattern.

In [12]:
# Query 1: affordable AND vegetarian

row1 = incidence_df.loc["affordable"]
row2 = incidence_df.loc["vegetarian"]

result = (row1 == 1) & (row2 == 1)

print("affordable AND vegetarian ->",
      incidence_df.columns[result].tolist())


# Try Query 2 and Query 3 below by changing the term names/operator.
row1 = incidence_df.loc["seafood"]
row2 = incidence_df.loc["beach"]

result = (row1 == 1) & (row2 == 1)

print("seafood AND beach ->",
      incidence_df.columns[result].tolist())
#query 3
row1 = incidence_df.loc["beach"]
row2 = incidence_df.loc["vegetarian"]

result = (row1 == 1) & (row2 == 0)

print("beach AND NOT vegetarian ->",
      incidence_df.columns[result].tolist())

affordable AND vegetarian -> ['D1']
seafood AND beach -> ['D3', 'D6']
beach AND NOT vegetarian -> ['D3', 'D6']


# Part 4 — Why do we need an inverted index?

The incidence matrix is easy to understand, but a real collection may have:

- hundreds of thousands of terms;
- millions of documents.

Most terms occur in only a small number of documents.

Therefore, most matrix cells are `0`.

This is called **sparsity**.

In [13]:
total_cells = incidence_df.size
ones = int(incidence_df.to_numpy().sum())
zeros = total_cells - ones

print("Total cells :", total_cells)
print("1 values    :", ones)
print("0 values    :", zeros)
print("Sparsity    :", round(zeros / total_cells, 3))

Total cells : 36
1 values    : 14
0 values    : 22
Sparsity    : 0.611


### Key idea

Instead of storing:

`vegetarian → [1, 1, 0, 0, 1, 0]`

we can store only the documents where the value is `1`:

`vegetarian → [D1, D2, D5]`

This is the central idea behind the **inverted index**.

# Part 5 — Build an inverted index

An **inverted index** maps:

> **term → documents containing that term**

Example:

`vegetarian → [D1, D2, D5]`

The list `[D1, D2, D5]` is the term's **postings list**.

Its length is the **document frequency (df)**:

`df(vegetarian) = 3`

## 5.1 First, predict some postings lists

Using the incidence matrix, write down the postings lists for:

- `affordable`
- `seafood`
- `coffee`

Then run the construction code below and compare.

## 5.2 How the construction works

For each document:

1. look at its processed terms;
2. use each term only once for this basic postings list;
3. add the document ID to that term's list.

Example idea:

```text
D1 contains vegetarian
→ add D1 to postings["vegetarian"]
```

### Your task

Complete only the line that adds the document ID to the term's postings list.

In [14]:
index = defaultdict(list)

for doc_id, tokens in processed_documents.items():

    for term in set(tokens):
        # TODO: add doc_id to this term's postings list
        index[term].append(doc_id)


inverted_index = dict(index)

for term in ["affordable", "vegetarian", "seafood", "beach", "coffee"]:
    postings = inverted_index.get(term, [])
    print(f"{term:12} -> {postings}   df={len(postings)}")

affordable   -> ['D1', 'D4']   df=2
vegetarian   -> ['D1', 'D2', 'D5']   df=3
seafood      -> ['D3', 'D5', 'D6']   df=3
beach        -> ['D1', 'D3', 'D6']   df=3
coffee       -> ['D4', 'D6']   df=2


## 5.3 Matrix row and postings list: same logical information

For `vegetarian`:

- incidence row tells us which columns contain `1`;
- postings list stores those document IDs directly.

Run this comparison.

In [15]:
term = "vegetarian"

matrix_docs = incidence_df.columns[
    incidence_df.loc[term] == 1
].tolist()

postings_docs = inverted_index.get(term, [])

print("From matrix :", matrix_docs)
print("From index  :", postings_docs)

From matrix : ['D1', 'D2', 'D5']
From index  : ['D1', 'D2', 'D5']


# Part 6 — Query processing using postings lists

Once the inverted index exists, we do not need to inspect the full matrix for every query.

For:

> `vegetarian AND beach`

retrieve:

- postings for `vegetarian`
- postings for `beach`

and find their intersection.

In [16]:
vegetarian_docs = set(inverted_index.get("vegetarian", []))
beach_docs = set(inverted_index.get("beach", []))

result = sorted(vegetarian_docs & beach_docs)

print("vegetarian AND beach ->", result)

vegetarian AND beach -> ['D1']


## 6.1 Boolean operations on postings

Python sets make the idea very clear:

- `A & B` → AND / intersection
- `A | B` → OR / union
- `A - B` → A AND NOT B

Complete the three small queries below.

In [17]:
# 1. seafood AND beach
A = set(inverted_index.get("seafood", []))
B = set(inverted_index.get("beach", []))
print("seafood AND beach ->", sorted(A & B))


# 2. coffee OR vegan
A = set(inverted_index.get("coffee", []))
B = set(inverted_index.get("vegan", []))
print("coffee OR vegan ->", sorted(A | B))


# 3. vegetarian AND NOT seafood
A = set(inverted_index.get("vegetarian", []))
B = set(inverted_index.get("seafood", []))
print("vegetarian AND NOT seafood ->", sorted(A - B))

seafood AND beach -> ['D3', 'D6']
coffee OR vegan -> ['D2', 'D4', 'D6']
vegetarian AND NOT seafood -> ['D1', 'D2']


## Optional: why are postings lists usually sorted?

Search systems normally store postings lists in sorted order.

For an AND query, two sorted lists can be intersected efficiently by moving through them from left to right.

You do **not** need to implement that algorithm in this practical.

The important idea for today is:

> Boolean AND on postings = **intersection of document lists**.

# Part 7 — A small end-to-end Boolean search system

We now combine preprocessing and the inverted index.

The function below is already provided.

It supports:

- one term;
- `term1 AND term2`;
- `term1 OR term2`;
- `term1 AND NOT term2`.

Your task is to **use and test the system**, not to write a query parser from scratch.

In [18]:
def normalise_one_term(text):
    tokens = preprocess(text)
    return tokens[0] if tokens else ""


def search(query):
    query = query.strip()

    if " AND NOT " in query:
        left, right = query.split(" AND NOT ", 1)
        A = set(inverted_index.get(normalise_one_term(left), []))
        B = set(inverted_index.get(normalise_one_term(right), []))
        return sorted(A - B)

    if " AND " in query:
        left, right = query.split(" AND ", 1)
        A = set(inverted_index.get(normalise_one_term(left), []))
        B = set(inverted_index.get(normalise_one_term(right), []))
        return sorted(A & B)

    if " OR " in query:
        left, right = query.split(" OR ", 1)
        A = set(inverted_index.get(normalise_one_term(left), []))
        B = set(inverted_index.get(normalise_one_term(right), []))
        return sorted(A | B)

    term = normalise_one_term(query)
    return inverted_index.get(term, [])


def show_results(query):
    matches = search(query)

    print("QUERY:", query)
    print("MATCHES:", matches)

    for doc_id in matches:
        print(f"  {doc_id}: {documents[doc_id]}")

In [19]:
show_results("vegetarian AND beach")

QUERY: vegetarian AND beach
MATCHES: ['D1']
  D1: Coastal Café near Malpe Beach serves affordable vegetarian meals.


## Final activity — Predict, search, check

For each query:

1. predict the document IDs;
2. run `show_results(...)`;
3. read the returned descriptions and check your prediction.

Try:

- `affordable AND vegetarian`
- `coffee OR vegan`
- `beach AND NOT vegetarian`
- `seafood AND beach`

Then create **one query of your own**.

In [20]:
show_results("affordable AND vegetarian")

# Run the remaining queries one by one below.

QUERY: affordable AND vegetarian
MATCHES: ['D1']
  D1: Coastal Café near Malpe Beach serves affordable vegetarian meals.


# Part 8 — What this search system cannot do

Consider the query:

> `cheap vegetarian food near seaside`

A relevant document may contain:

> `affordable vegetarian meals near the beach`

Our current system does not automatically know that:

- `cheap` is related to `affordable`;
- `seaside` is related to `beach`;
- `food` may be related to `meals`.

Boolean retrieval is therefore:

- **exact**;
- easy to understand;
- efficient with an inverted index;
- but limited when the user and document use different words.

Later retrieval methods address some of these limitations.

## Short reflection / viva

Answer in one or two sentences each:

1. What does `1` mean in an incidence matrix?
2. Why does an incidence matrix contain many zeros in a large collection?
3. What is a postings list?
4. What does `df` mean?
5. How is an incidence row related to a postings list?
6. How is Boolean AND performed using postings lists?
7. Why must documents and queries use compatible preprocessing?
8. Give one limitation of exact Boolean retrieval.

### Before you finish

Make sure you can explain this progression:

> **incidence matrix → many zeros → inverted index → postings lists → Boolean query processing**